# 13 — Structural breaks

Test pre-specified breaks at 2013, 2021 and 2022. With only 20 annual observations, keep the model deliberately small and treat break tests as diagnostics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.breaks import chow_test, interrupted_time_series


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
transition_years = (2021,)
records = []
for product, sub in panel.groupby("product"):
    for outcome in ["exports_kt", "imports_kt", "refinery_output_kt", "net_import_to_demand_ratio"]:
        if outcome not in sub or sub[outcome].notna().sum() < 10:
            continue
        for break_year in [2013, 2022]:
            try:
                result = chow_test(sub["year"], sub[outcome], break_year=break_year, transition_years=transition_years)
            except ValueError:
                continue
            records.append({
                "product": product, "outcome": outcome, "break_year": break_year,
                "f_statistic": result.f_statistic, "p_value": result.p_value,
                "n_pre": result.n_pre, "n_post": result.n_post,
                "excluded_years": ",".join(str(year) for year in result.excluded_years),
                "method": "pre-specified Chow linear-trend diagnostic",
            })
breaks = pd.DataFrame(records)
if not breaks.empty:
    breaks = breaks.sort_values("p_value").reset_index(drop=True)
    m = len(breaks)
    raw_bh = breaks["p_value"] * m / (breaks.index + 1)
    breaks["bh_fdr_p_value"] = raw_bh.iloc[::-1].cummin().iloc[::-1].clip(upper=1.0)
persist_dataframe(breaks, PATHS.metrics / "structural_break_tests.csv")
display(breaks.sort_values("p_value").head(20))


In [ ]:
# Event-aligned interrupted trend models: coefficients are saved, not copied by hand.
coef_rows = []
for product, sub in panel.groupby("product"):
    for outcome in ["exports_kt", "net_import_to_demand_ratio"]:
        if outcome not in sub or sub[outcome].notna().sum() < 10:
            continue
        for event_year in [2013, 2022]:
            model = interrupted_time_series(sub, value_column=outcome, event_year=event_year, transition_years=transition_years)
            for term in model.params.index:
                coef_rows.append({
                    "product": product, "outcome": outcome, "event_year": event_year,
                    "term": term, "estimate": model.params[term], "std_error": model.bse[term],
                    "p_value": model.pvalues[term], "nobs": model.nobs,
                    "covariance": "HAC(1)", "interpretation": "associational interrupted trend",
                })
coef = pd.DataFrame(coef_rows)
persist_dataframe(coef, PATHS.metrics / "annual_interrupted_trend_models.csv")
display(coef.head(20))
